# FIT3182 Assignment 2 — Data Visualisation

## Task 2.2.1: Violation Counts & Speed Patterns  
## Task 2.2.2: Interesting Point Annotation

This notebook queries MongoDB for violation data and produces **interactive** (Plotly) and
**static** (Matplotlib) visualisations with annotated interesting points.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from pymongo import MongoClient
from datetime import datetime, timedelta
from plotly.subplots import make_subplots

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

# Srt plot size
plt.rcParams['figure.figsize'] = (14, 6)

# Set font size within plot
plt.rcParams['font.size'] = 11

# Intialise params
mg_db_url = "mongodb://localhost:27017/"
db_name = "AWAS_speed_violations"

# Initialise DB
client = MongoClient(mg_db_url)
db = client[db_name]

# print(f"Connected to: {db_name}")
# print(f"Violation documents: {db.violations.count_documents({})}")

In [ ]:
pipeline = [
    {"$unwind": "$violations"},
    {"$project": {
        "car_plate": 1,
        "date": 1,
        "violation_type": "$violations.violation_type",
        "camera_id_start": "$violations.camera_id_start",
        "camera_id_end": "$violations.camera_id_end",
        "timestamp_start": "$violations.timestamp_start",
        "timestamp_end": "$violations.timestamp_end",
        "speed_reading": "$violations.speed_reading",
        "speed_limit": "$violations.speed_limit",
        "excess_kmh": "$violations.excess_kmh"
    }}
]

records = list(db.violations.aggregate(pipeline, allowDiskUse = True))

if records:
    viols_df = pd.DataFrame(records)
    viols_df['date'] = pd.to_datetime(viols_df['date'])

    """print(f"Extracted {len(viols_df)} individual violations")
    print(f"Types: {viols_df['violation_type'].value_counts().to_dict()}")
    print(f"Date range: {viols_df['date'].min()} → {viols_df['date'].max()}")"""
else:
    # Could consider removing entire section below

    # Generate sample data for demonstration if no streaming data yet
    print("No MongoDB data found — generating sample for demonstration.")
    np.random.seed(42)
    dates = pd.date_range('2024-01-01', '2024-08-27', freq='D')
    rows = []
    for d in dates:
        for _ in range(np.random.poisson(15)):
            rows.append({
                'date': d, 'violation_type': 'INSTANTANEOUS',
                'speed_reading': 110 + np.random.exponential(15),
                'speed_limit': np.random.choice([110, 110, 90]),
                'excess_kmh': np.random.exponential(15),
                'timestamp_start': d + timedelta(hours=8 + np.random.exponential(3)),
                'camera_id_start': np.random.choice([1, 2, 3]),
                'camera_id_end': np.random.choice([1, 2, 3])
            })
        for _ in range(np.random.poisson(8)):
            rows.append({
                'date': d, 'violation_type': 'AVERAGE',
                'speed_reading': 90 + np.random.exponential(20),
                'speed_limit': np.random.choice([110, 90]),
                'excess_kmh': np.random.exponential(20),
                'timestamp_end': d + timedelta(hours=8 + np.random.exponential(3)),
                'camera_id_start': np.random.choice([1, 2]),
                'camera_id_end': np.random.choice([2, 3])
            })
    viols_df = pd.DataFrame(rows)
    viols_df['date'] = pd.to_datetime(viols_df['date'])
    print(f"Generated {len(viols_df)} sample violations")


---
## Visualisation 1 — Interactive: Daily Violation Counts Over Time (Plotly)

This interactive chart shows daily violation counts split by type. Users can zoom, pan,
hover for details, and toggle series. The 7-day moving average reveals underlying trends.


In [ ]:
# ============================================================
# INTERACTIVE PLOT 1: Daily Violation Counts (Plotly)
# ============================================================
daily = viols_df.groupby([pd.Grouper(key='date', freq='D'), 'violation_type']).size().unstack(fill_value=0)
for c in ['INSTANTANEOUS', 'AVERAGE']:
    if c not in daily.columns:
        daily[c] = 0
daily['total'] = daily.sum(axis=1)
daily['MA_7'] = daily['total'].rolling(7, center=True).mean()

fig = make_subplots(specs=[[{"secondary_y": False}]])

fig.add_trace(go.Scatter(
    x=daily.index, y=daily['INSTANTANEOUS'],
    name='Instantaneous', mode='lines',
    line=dict(color='#E74C3C', width=1.5),
    hovertemplate='Date: %{x}<br>Instant: %{y}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=daily.index, y=daily['AVERAGE'],
    name='Average Speed', mode='lines',
    line=dict(color='#3498DB', width=1.5),
    hovertemplate='Date: %{x}<br>Average: %{y}<extra></extra>'
))
fig.add_trace(go.Scatter(
    x=daily.index, y=daily['total'],
    name='Total', mode='lines',
    line=dict(color='#2C3E50', width=1, dash='dot'),
    opacity=0.5
))
fig.add_trace(go.Scatter(
    x=daily.index, y=daily['MA_7'],
    name='7-Day Moving Avg', mode='lines',
    line=dict(color='orange', width=2.5)
))

# Annotate peak
peak_idx = daily['total'].idxmax()
peak_val = daily['total'].max()
fig.add_annotation(
    x=peak_idx, y=peak_val,
    text=f"PEAK: {peak_val:.0f}<br>{peak_idx.strftime('%Y-%m-%d')}",
    showarrow=True, arrowhead=2, arrowcolor='red',
    bgcolor='yellow', font=dict(size=11, color='red')
)

# Annotate minimum (non-zero)
nonzero = daily[daily['total'] > 0]['total']
if len(nonzero) > 0:
    min_idx = nonzero.idxmin()
    min_val = nonzero.min()
    fig.add_annotation(
        x=min_idx, y=min_val,
        text=f"MIN: {min_val:.0f}<br>{min_idx.strftime('%Y-%m-%d')}",
        showarrow=True, arrowhead=2, arrowcolor='green',
        bgcolor='lightgreen', font=dict(size=11, color='green')
    )

fig.update_layout(
    title='Daily Violation Counts Over Time (Interactive)',
    xaxis_title='Date', yaxis_title='Violation Count',
    hovermode='x unified', template='plotly_white',
    height=500, legend=dict(orientation='h', y=1.1)
)
fig.show()


---
## Visualisation 2 — Interactive: Real-time Violation Speed with Dynamic Analysis (Plotly)

Shows violation speeds over arrival time with percentile bands, spike/drop detection,
and a rolling moving average — inspired by the starting guide's sample plot.


In [ ]:
# ============================================================
# INTERACTIVE PLOT 2: Speed Distribution with Dynamic Analysis (Plotly)
# ============================================================

# Get timestamp for each violation
viols_df_ts = viols_df.copy()
if 'timestamp_start' in viols_df_ts.columns:
    viols_df_ts['ts'] = viols_df_ts['timestamp_start'].combine_first(
        viols_df_ts.get('timestamp_end', pd.Series(dtype='datetime64[ns]'))
    )
else:
    viols_df_ts['ts'] = viols_df_ts['date']

viols_df_ts = viols_df_ts.dropna(subset=['ts'])
viols_df_ts = viols_df_ts.sort_values('ts')

# Compute rolling statistics (5-minute window by index position)
window_size = min(50, len(viols_df_ts))
viols_df_ts['rolling_mean'] = viols_df_ts['speed_reading'].rolling(window_size, center=True).mean()
viols_df_ts['rolling_std'] = viols_df_ts['speed_reading'].rolling(window_size, center=True).std()
viols_df_ts['rolling_median'] = viols_df_ts['speed_reading'].rolling(window_size, center=True).median()

# Percentile coloring
p50 = viols_df_ts['speed_reading'].quantile(0.50)
p75 = viols_df_ts['speed_reading'].quantile(0.75)
p90 = viols_df_ts['speed_reading'].quantile(0.90)

def speed_category(s):
    if s > p90: return 'Very High (>90th pctl)'
    elif s > p75: return 'High (75-90th pctl)'
    elif s > p50: return 'Medium (50-75th pctl)'
    else: return 'Low (<50th pctl)'

viols_df_ts['category'] = viols_df_ts['speed_reading'].apply(speed_category)

color_map = {
    'Very High (>90th pctl)': '#E74C3C',
    'High (75-90th pctl)': '#F39C12',
    'Medium (50-75th pctl)': '#F1C40F',
    'Low (<50th pctl)': '#27AE60'
}

# Sample for performance (max 2000 points)
if len(viols_df_ts) > 2000:
    plot_df = viols_df_ts.sample(2000, random_state=42).sort_values('ts')
else:
    plot_df = viols_df_ts

fig2 = go.Figure()

# Scatter by category
for cat, color in color_map.items():
    mask = plot_df['category'] == cat
    fig2.add_trace(go.Scatter(
        x=plot_df[mask]['ts'], y=plot_df[mask]['speed_reading'],
        mode='markers', name=cat,
        marker=dict(color=color, size=5, opacity=0.7),
        hovertemplate='Time: %{x}<br>Speed: %{y:.1f} km/h<extra></extra>'
    ))

# Moving average line
fig2.add_trace(go.Scatter(
    x=plot_df['ts'], y=plot_df['rolling_mean'],
    mode='lines', name='Moving Average',
    line=dict(color='purple', width=2.5)
))

# Annotate MAX and MIN
max_row = viols_df_ts.loc[viols_df_ts['speed_reading'].idxmax()]
min_row = viols_df_ts.loc[viols_df_ts['speed_reading'].idxmin()]

fig2.add_annotation(
    x=max_row['ts'], y=max_row['speed_reading'],
    text=f"MAX: {max_row['speed_reading']:.1f} km/h<br>({p90:.0f}+ pctl)",
    showarrow=True, arrowhead=2, bgcolor='red',
    font=dict(color='white', size=10)
)
fig2.add_annotation(
    x=min_row['ts'], y=min_row['speed_reading'],
    text=f"MIN: {min_row['speed_reading']:.1f} km/h",
    showarrow=True, arrowhead=2, bgcolor='green',
    font=dict(color='white', size=10)
)

# Detect spikes and drops
if 'rolling_median' in viols_df_ts.columns:
    spikes = plot_df[plot_df['speed_reading'] > plot_df['rolling_median'] * 1.15]
    drops = plot_df[plot_df['speed_reading'] < plot_df['rolling_median'] * 0.80]

    if len(spikes) > 0:
        top_spike = spikes.nlargest(1, 'speed_reading').iloc[0]
        pct_above = ((top_spike['speed_reading'] / top_spike['rolling_median']) - 1) * 100
        fig2.add_annotation(
            x=top_spike['ts'], y=top_spike['speed_reading'],
            text=f"SPIKE!<br>{top_spike['speed_reading']:.1f} km/h<br>({pct_above:.0f}% above median)",
            showarrow=True, bgcolor='#E74C3C', font=dict(color='white', size=9)
        )

    if len(drops) > 0:
        top_drop = drops.nsmallest(1, 'speed_reading').iloc[0]
        pct_below = (1 - top_drop['speed_reading'] / top_drop['rolling_median']) * 100
        fig2.add_annotation(
            x=top_drop['ts'], y=top_drop['speed_reading'],
            text=f"DROP!<br>{top_drop['speed_reading']:.1f} km/h<br>({pct_below:.0f}% below median)",
            showarrow=True, bgcolor='#3498DB', font=dict(color='white', size=9)
        )

# Percentile reference lines
for pval, plabel, pcolor in [(p50, 'Median', '#7F8C8D'),
                              (p75, '75th', '#F39C12'),
                              (p90, '90th', '#E74C3C')]:
    fig2.add_hline(y=pval, line_dash='dash', line_color=pcolor, opacity=0.4,
                   annotation_text=f"{plabel}: {pval:.0f}", annotation_position='right')

# Stats box
stats_text = (f"Count: {len(viols_df_ts)} | Mean: {viols_df_ts['speed_reading'].mean():.1f} | "
              f"Std: {viols_df_ts['speed_reading'].std():.1f} | "
              f"Range: {viols_df_ts['speed_reading'].max() - viols_df_ts['speed_reading'].min():.1f}")

fig2.update_layout(
    title=f'Real-time Violation Speeds with Dynamic Analysis<br><sub>{stats_text}</sub>',
    xaxis_title='Time', yaxis_title='Speed (km/h)',
    hovermode='closest', template='plotly_white', height=550,
    legend=dict(orientation='h', y=-0.15)
)
fig2.show()


---
## Visualisation 3 — Interactive: Violation Count & Average Speed (Dual Axis, Plotly)

Inspired by the starting guide's dual-axis sample plot — shows violation count and average
violation speed over rolling time windows with annotated maxima and minima.


In [ ]:
# ============================================================
# INTERACTIVE PLOT 3: Dual-Axis — Violations Count + Avg Speed (Plotly)
# ============================================================
# Group by date
daily_stats = viols_df.groupby(pd.Grouper(key='date', freq='D')).agg(
    count=('speed_reading', 'size'),
    avg_speed=('speed_reading', 'mean'),
    max_speed=('speed_reading', 'max')
).dropna()

fig3 = make_subplots(specs=[[{"secondary_y": True}]])

# Violation count (left axis)
fig3.add_trace(
    go.Scatter(x=daily_stats.index, y=daily_stats['count'],
               name='Violations Count', mode='lines+markers',
               line=dict(color='#3498DB', width=2),
               marker=dict(size=4)),
    secondary_y=False
)

# Average speed (right axis)
fig3.add_trace(
    go.Scatter(x=daily_stats.index, y=daily_stats['avg_speed'],
               name='Average Speed', mode='lines+markers',
               line=dict(color='#E74C3C', width=2),
               marker=dict(size=4, symbol='square')),
    secondary_y=True
)

# Annotate count peak
max_count_idx = daily_stats['count'].idxmax()
max_count_val = daily_stats['count'].max()
fig3.add_annotation(
    x=max_count_idx, y=max_count_val,
    text=f"MAX: {max_count_val:.0f}",
    showarrow=True, arrowhead=2, bgcolor='yellow',
    font=dict(size=11, color='blue')
)

# Annotate speed peak
max_spd_idx = daily_stats['avg_speed'].idxmax()
max_spd_val = daily_stats['avg_speed'].max()
fig3.add_annotation(
    x=max_spd_idx, y=max_spd_val,
    text=f"MAX: {max_spd_val:.1f} km/h",
    showarrow=True, arrowhead=2, bgcolor='#FADBD8',
    font=dict(size=11, color='red'),
    yref='y2'
)

# Annotate speed minimum
min_spd_idx = daily_stats['avg_speed'].idxmin()
min_spd_val = daily_stats['avg_speed'].min()
fig3.add_annotation(
    x=min_spd_idx, y=min_spd_val,
    text=f"MIN: {min_spd_val:.1f} km/h",
    showarrow=True, arrowhead=2, bgcolor='lightgreen',
    font=dict(size=11, color='green'),
    yref='y2'
)

fig3.update_layout(
    title='Real-time Violations & Average Speed',
    xaxis_title='Date',
    hovermode='x unified', template='plotly_white', height=500,
    legend=dict(orientation='h', y=1.1)
)
fig3.update_yaxes(title_text='Violation Count', secondary_y=False)
fig3.update_yaxes(title_text='Average Speed (km/h)', secondary_y=True)
fig3.show()


---
## Visualisation 4 — Static: Camera & Segment Violation Breakdown (Matplotlib)


In [ ]:
# ============================================================
# STATIC PLOT 4: Camera / Segment Breakdown (Matplotlib)
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Instantaneous by camera
instant = viols_df[viols_df['violation_type'] == 'INSTANTANEOUS']
if 'camera_id_start' in instant.columns and len(instant) > 0:
    cam_counts = instant.groupby('camera_id_start').size()
    limits_labels = {1: '110', 2: '110', 3: '90'}
    colors = ['#E74C3C', '#F39C12', '#8E44AD']
    bars = ax1.bar([f"Camera {c}\n(limit {limits_labels.get(c, '?')} km/h)"
                    for c in cam_counts.index],
                   cam_counts.values, color=colors[:len(cam_counts)], alpha=0.85)
    for bar, val in zip(bars, cam_counts.values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{val:,}', ha='center', fontweight='bold', fontsize=11)
    # Highlight max
    max_idx = cam_counts.values.argmax()
    bars[max_idx].set_edgecolor('red')
    bars[max_idx].set_linewidth(3)

ax1.set_title('Instantaneous Violations by Camera', fontsize=13, fontweight='bold')
ax1.set_ylabel('Count')

# Average speed by segment
avg_v = viols_df[viols_df['violation_type'] == 'AVERAGE']
if 'camera_id_start' in avg_v.columns and 'camera_id_end' in avg_v.columns and len(avg_v) > 0:
    avg_v = avg_v.copy()
    avg_v['segment'] = avg_v.apply(
        lambda r: f"{int(r['camera_id_start'])}→{int(r['camera_id_end'])}", axis=1)
    seg_counts = avg_v.groupby('segment').size()
    seg_colors = ['#3498DB', '#1ABC9C']
    bars2 = ax2.bar(seg_counts.index, seg_counts.values,
                    color=seg_colors[:len(seg_counts)], alpha=0.85)
    for bar, val in zip(bars2, seg_counts.values):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val:,}', ha='center', fontweight='bold', fontsize=11)

ax2.set_title('Average Speed Violations by Segment', fontsize=13, fontweight='bold')
ax2.set_ylabel('Count')
ax2.set_xlabel('Segment (Start → End Camera)')

plt.tight_layout()
plt.savefig('../outputs/camera_segment_violations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: camera_segment_violations.png")


---
## Visualisation 5 — Interactive: Hourly Heatmap (Plotly)

Shows violation intensity by hour and day of week — helps identify enforcement scheduling
opportunities.


In [ ]:
# ============================================================
# INTERACTIVE PLOT 5: Hourly/Day Heatmap (Plotly)
# ============================================================
viol_ts = viols_df.copy()
if 'timestamp_start' in viol_ts.columns:
    viol_ts['ts'] = viol_ts['timestamp_start'].combine_first(
        viol_ts.get('timestamp_end', pd.Series(dtype='datetime64[ns]'))
    )
else:
    viol_ts['ts'] = viol_ts['date']

viol_ts = viol_ts.dropna(subset=['ts'])
viol_ts['hour'] = pd.to_datetime(viol_ts['ts']).dt.hour
viol_ts['dayofweek'] = pd.to_datetime(viol_ts['ts']).dt.day_name()

heatmap_data = viol_ts.groupby(['dayofweek', 'hour']).size().unstack(fill_value=0)

# Reorder days
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex([d for d in day_order if d in heatmap_data.index])

fig5 = px.imshow(
    heatmap_data.values,
    labels=dict(x='Hour of Day', y='Day of Week', color='Violations'),
    x=[str(h) for h in heatmap_data.columns],
    y=list(heatmap_data.index),
    color_continuous_scale='YlOrRd',
    aspect='auto'
)
fig5.update_layout(
    title='Violation Heatmap: Hour of Day × Day of Week',
    height=400, template='plotly_white'
)
fig5.show()


---
## Task 2.2.2 — Operational Significance of Annotations

### Why These Visualisations Matter

| Visualisation | Annotations | Operational Value |
|---|---|---|
| **Daily violation count** | Peak day, minimum day, 7-day MA | Identifies high-violation dates for investigation (holiday? road works?). The moving average reveals trend direction — are violations increasing or decreasing over months? |
| **Violation speeds with dynamic analysis** | MAX/MIN, spike/drop detection, percentile bands | Identifies the most dangerous offenders (top percentile) and timing. Spikes above the rolling median signal unusual events. Drops may indicate camera outages or reduced traffic. |
| **Dual-axis count + speed** | MAX count, MAX/MIN speed | Reveals whether high-count days also have high-speed violations, or if the two patterns diverge. Divergence may indicate different root causes. |
| **Camera/segment breakdown** | Highlighted max camera | Camera 3 (90 km/h limit) likely captures disproportionate violations. This informs whether the limit needs review or better advance warning signage. |
| **Hourly heatmap** | Color intensity | Pinpoints exact hours and days for patrol scheduling. Darkest cells = highest enforcement ROI windows. |

### Recommendations
1. **Deploy enforcement** during peak hours identified in the heatmap.
2. **Investigate** sudden spike dates — correlate with incidents, weather, or road changes.
3. **Review Camera 3 signage** if its violation rate is disproportionately high due to the 90 km/h limit.
4. **Track the 7-day MA trend** to evaluate whether enforcement actions reduce violations over time.
5. **Prioritize "Very High" percentile** offenders for targeted enforcement.


In [ ]:
# Cleanup
client.close()
print("MongoDB connection closed. All visualisations complete.")
